In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import timm


from paddy_10_data_loader import load_train_val_test_data
from shufflenet_v2 import ShuffleNetV2
from helper_utils import show_sample_images, training_loop, plot_training_history, visualize_predictions, plot_confusion_matrix

import kd_utils

from fit_net_wrapper import FitNetWrapper

In [2]:
student_train_loader, student_val_loader, student_test_loader = load_train_val_test_data(batch_size=32)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
densenet_201_teacher = timm.create_model("densenet201", pretrained=False, num_classes=10)
densenet_201_teacher.load_state_dict(torch.load("densenet_201_teacher.pth"))

<All keys matched successfully>

# Train with FitNet

In [5]:
hint_epochs = 15
train_epochs = 15

In [7]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=3.0,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2388
Epoch 2/15 - Hint Loss: 1.7792
Epoch 3/15 - Hint Loss: 1.6163
Epoch 4/15 - Hint Loss: 1.5353
Epoch 5/15 - Hint Loss: 1.4853
Epoch 6/15 - Hint Loss: 1.4507
Epoch 7/15 - Hint Loss: 1.4260
Epoch 8/15 - Hint Loss: 1.4067
Epoch 9/15 - Hint Loss: 1.3917
Epoch 10/15 - Hint Loss: 1.3937
Epoch 11/15 - Hint Loss: 1.3779
Epoch 12/15 - Hint Loss: 1.3693
Epoch 13/15 - Hint Loss: 1.3638
Epoch 14/15 - Hint Loss: 1.3583
Epoch 15/15 - Hint Loss: 1.3548


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.6860%, Train Hard Loss: 1.3041, Train Soft Loss: 0.9125, Train Distill Loss: 2.6858 | Val Hard Loss: 1.4689, Val Soft Loss: 0.4523, Val Distill Loss: 4.3531, Val Acc: 61.9904%
Epoch 2/15 - Train Acc: 75.8978%, Train Hard Loss: 0.7972, Train Soft Loss: 0.5133, Train Distill Loss: 1.5618 | Val Hard Loss: 0.8663, Val Soft Loss: 0.2723, Val Distill Loss: 2.6113, Val Acc: 76.0791%
Epoch 3/15 - Train Acc: 82.2239%, Train Hard Loss: 0.5938, Train Soft Loss: 0.3822, Train Distill Loss: 1.1630 | Val Hard Loss: 0.7236, Val Soft Loss: 0.2690, Val Distill Loss: 2.5140, Val Acc: 80.0360%
Epoch 4/15 - Train Acc: 88.1443%, Train Hard Loss: 0.3803, Train Soft Loss: 0.2687, Train Distill Loss: 0.7879 | Val Hard Loss: 0.8979, Val Soft Loss: 0.3047, Val Distill Loss: 2.8867, Val Acc: 78.4173%
Epoch 5/15 - Train Acc: 91.7355%, Train Hard Loss: 0.2623, Train Soft Loss: 0.2085, Train Distill Loss: 0.5852 | Val Hard Loss: 0.4775, Val Soft Loss: 0.1639, Val Distill Loss: 1.5504, Val

0.9333013435700576

In [8]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=3.5,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2697
Epoch 2/15 - Hint Loss: 1.7863
Epoch 3/15 - Hint Loss: 1.6223
Epoch 4/15 - Hint Loss: 1.5390
Epoch 5/15 - Hint Loss: 1.4878
Epoch 6/15 - Hint Loss: 1.4515
Epoch 7/15 - Hint Loss: 1.4241
Epoch 8/15 - Hint Loss: 1.4040
Epoch 9/15 - Hint Loss: 1.3897
Epoch 10/15 - Hint Loss: 1.3785
Epoch 11/15 - Hint Loss: 1.3717
Epoch 12/15 - Hint Loss: 1.3640
Epoch 13/15 - Hint Loss: 1.3585
Epoch 14/15 - Hint Loss: 1.3539
Epoch 15/15 - Hint Loss: 1.3522


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 58.4974%, Train Hard Loss: 1.3181, Train Soft Loss: 0.7789, Train Distill Loss: 2.9629 | Val Hard Loss: 1.3348, Val Soft Loss: 0.4209, Val Distill Loss: 4.0349, Val Acc: 66.0072%
Epoch 2/15 - Train Acc: 75.6724%, Train Hard Loss: 0.8318, Train Soft Loss: 0.4440, Train Distill Loss: 1.7532 | Val Hard Loss: 0.9551, Val Soft Loss: 0.2826, Val Distill Loss: 2.7382, Val Acc: 75.4197%
Epoch 3/15 - Train Acc: 83.6965%, Train Hard Loss: 0.5569, Train Soft Loss: 0.3090, Train Distill Loss: 1.2026 | Val Hard Loss: 0.8349, Val Soft Loss: 0.2802, Val Distill Loss: 2.6594, Val Acc: 77.8777%
Epoch 4/15 - Train Acc: 88.5650%, Train Hard Loss: 0.3741, Train Soft Loss: 0.2297, Train Distill Loss: 0.8620 | Val Hard Loss: 1.5830, Val Soft Loss: 0.3806, Val Distill Loss: 3.8365, Val Acc: 68.5851%
Epoch 5/15 - Train Acc: 91.8858%, Train Hard Loss: 0.2526, Train Soft Loss: 0.1819, Train Distill Loss: 0.6477 | Val Hard Loss: 0.5394, Val Soft Loss: 0.1767, Val Distill Loss: 1.6833, Val

0.9328214971209213

In [9]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4.0,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2295
Epoch 2/15 - Hint Loss: 1.7690
Epoch 3/15 - Hint Loss: 1.6175
Epoch 4/15 - Hint Loss: 1.5415
Epoch 5/15 - Hint Loss: 1.4898
Epoch 6/15 - Hint Loss: 1.4576
Epoch 7/15 - Hint Loss: 1.4296
Epoch 8/15 - Hint Loss: 1.4109
Epoch 9/15 - Hint Loss: 1.3949
Epoch 10/15 - Hint Loss: 1.3827
Epoch 11/15 - Hint Loss: 1.3730
Epoch 12/15 - Hint Loss: 1.3660
Epoch 13/15 - Hint Loss: 1.3599
Epoch 14/15 - Hint Loss: 1.3622
Epoch 15/15 - Hint Loss: 1.3534


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.4906%, Train Hard Loss: 1.3593, Train Soft Loss: 0.6524, Train Distill Loss: 3.1753 | Val Hard Loss: 1.3433, Val Soft Loss: 0.4039, Val Distill Loss: 3.9028, Val Acc: 66.4269%
Epoch 2/15 - Train Acc: 75.6274%, Train Hard Loss: 0.8600, Train Soft Loss: 0.3724, Train Distill Loss: 1.8796 | Val Hard Loss: 1.0658, Val Soft Loss: 0.3389, Val Distill Loss: 3.2440, Val Acc: 71.5827%
Epoch 3/15 - Train Acc: 84.0120%, Train Hard Loss: 0.5431, Train Soft Loss: 0.2539, Train Distill Loss: 1.2469 | Val Hard Loss: 0.6548, Val Soft Loss: 0.2182, Val Distill Loss: 2.0731, Val Acc: 81.4748%
Epoch 4/15 - Train Acc: 88.9256%, Train Hard Loss: 0.3575, Train Soft Loss: 0.1907, Train Distill Loss: 0.8961 | Val Hard Loss: 0.6129, Val Soft Loss: 0.1882, Val Distill Loss: 1.8121, Val Acc: 84.1727%
Epoch 5/15 - Train Acc: 92.8625%, Train Hard Loss: 0.2242, Train Soft Loss: 0.1475, Train Distill Loss: 0.6515 | Val Hard Loss: 0.4705, Val Soft Loss: 0.1487, Val Distill Loss: 1.4248, Val

0.9433781190019194

In [10]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=4.5,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2484
Epoch 2/15 - Hint Loss: 1.7885
Epoch 3/15 - Hint Loss: 1.6221
Epoch 4/15 - Hint Loss: 1.5365
Epoch 5/15 - Hint Loss: 1.4847
Epoch 6/15 - Hint Loss: 1.4505
Epoch 7/15 - Hint Loss: 1.4254
Epoch 8/15 - Hint Loss: 1.4061
Epoch 9/15 - Hint Loss: 1.3909
Epoch 10/15 - Hint Loss: 1.3790
Epoch 11/15 - Hint Loss: 1.3731
Epoch 12/15 - Hint Loss: 1.3638
Epoch 13/15 - Hint Loss: 1.3583
Epoch 14/15 - Hint Loss: 1.3549
Epoch 15/15 - Hint Loss: 1.3512


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.4155%, Train Hard Loss: 1.3592, Train Soft Loss: 0.5435, Train Distill Loss: 3.2887 | Val Hard Loss: 1.1345, Val Soft Loss: 0.3929, Val Distill Loss: 3.7104, Val Acc: 65.2878%
Epoch 2/15 - Train Acc: 77.0548%, Train Hard Loss: 0.7937, Train Soft Loss: 0.2971, Train Distill Loss: 1.8381 | Val Hard Loss: 1.0099, Val Soft Loss: 0.3009, Val Distill Loss: 2.9123, Val Acc: 74.5803%
Epoch 3/15 - Train Acc: 83.8167%, Train Hard Loss: 0.5395, Train Soft Loss: 0.2107, Train Distill Loss: 1.2847 | Val Hard Loss: 0.8487, Val Soft Loss: 0.2603, Val Distill Loss: 2.5064, Val Acc: 78.1175%
Epoch 4/15 - Train Acc: 88.9557%, Train Hard Loss: 0.3671, Train Soft Loss: 0.1602, Train Distill Loss: 0.9423 | Val Hard Loss: 0.6016, Val Soft Loss: 0.1738, Val Distill Loss: 1.6914, Val Acc: 85.0719%
Epoch 5/15 - Train Acc: 92.7874%, Train Hard Loss: 0.2341, Train Soft Loss: 0.1276, Train Distill Loss: 0.7042 | Val Hard Loss: 0.3926, Val Soft Loss: 0.1245, Val Distill Loss: 1.1925, Val

0.935700575815739

In [11]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=5.0,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2538
Epoch 2/15 - Hint Loss: 1.7854
Epoch 3/15 - Hint Loss: 1.6223
Epoch 4/15 - Hint Loss: 1.5358
Epoch 5/15 - Hint Loss: 1.4843
Epoch 6/15 - Hint Loss: 1.4509
Epoch 7/15 - Hint Loss: 1.4274
Epoch 8/15 - Hint Loss: 1.4069
Epoch 9/15 - Hint Loss: 1.3915
Epoch 10/15 - Hint Loss: 1.3800
Epoch 11/15 - Hint Loss: 1.3716
Epoch 12/15 - Hint Loss: 1.3641
Epoch 13/15 - Hint Loss: 1.3588
Epoch 14/15 - Hint Loss: 1.3546
Epoch 15/15 - Hint Loss: 1.3511


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.5507%, Train Hard Loss: 1.3611, Train Soft Loss: 0.4514, Train Distill Loss: 3.3460 | Val Hard Loss: 2.5802, Val Soft Loss: 0.6716, Val Distill Loss: 6.6630, Val Acc: 50.3597%
Epoch 2/15 - Train Acc: 76.7243%, Train Hard Loss: 0.8165, Train Soft Loss: 0.2440, Train Distill Loss: 1.8734 | Val Hard Loss: 0.9356, Val Soft Loss: 0.2584, Val Distill Loss: 2.5353, Val Acc: 75.1799%
Epoch 3/15 - Train Acc: 84.4929%, Train Hard Loss: 0.5199, Train Soft Loss: 0.1733, Train Distill Loss: 1.2826 | Val Hard Loss: 1.0357, Val Soft Loss: 0.2808, Val Distill Loss: 2.7640, Val Acc: 74.2206%
Epoch 4/15 - Train Acc: 88.6401%, Train Hard Loss: 0.3726, Train Soft Loss: 0.1368, Train Distill Loss: 0.9820 | Val Hard Loss: 1.0805, Val Soft Loss: 0.2767, Val Distill Loss: 2.7542, Val Acc: 75.8393%
Epoch 5/15 - Train Acc: 92.4869%, Train Hard Loss: 0.2291, Train Soft Loss: 0.1095, Train Distill Loss: 0.7310 | Val Hard Loss: 0.5126, Val Soft Loss: 0.1564, Val Distill Loss: 1.5077, Val

0.9285028790786948

In [12]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=5.5,
    alpha=0.8,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2166
Epoch 2/15 - Hint Loss: 1.7681
Epoch 3/15 - Hint Loss: 1.6148
Epoch 4/15 - Hint Loss: 1.5346
Epoch 5/15 - Hint Loss: 1.4851
Epoch 6/15 - Hint Loss: 1.4508
Epoch 7/15 - Hint Loss: 1.4306
Epoch 8/15 - Hint Loss: 1.4095
Epoch 9/15 - Hint Loss: 1.3941
Epoch 10/15 - Hint Loss: 1.3826
Epoch 11/15 - Hint Loss: 1.3750
Epoch 12/15 - Hint Loss: 1.3671
Epoch 13/15 - Hint Loss: 1.3614
Epoch 14/15 - Hint Loss: 1.3569
Epoch 15/15 - Hint Loss: 1.3532


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 57.8963%, Train Hard Loss: 1.3626, Train Soft Loss: 0.3939, Train Distill Loss: 3.4730 | Val Hard Loss: 0.9596, Val Soft Loss: 0.3408, Val Distill Loss: 3.2060, Val Acc: 71.5228%
Epoch 2/15 - Train Acc: 76.5440%, Train Hard Loss: 0.8462, Train Soft Loss: 0.2181, Train Distill Loss: 1.9963 | Val Hard Loss: 1.2166, Val Soft Loss: 0.3418, Val Distill Loss: 3.3428, Val Acc: 73.0815%
Epoch 3/15 - Train Acc: 84.8835%, Train Hard Loss: 0.5300, Train Soft Loss: 0.1517, Train Distill Loss: 1.3419 | Val Hard Loss: 0.5463, Val Soft Loss: 0.1805, Val Distill Loss: 1.7168, Val Acc: 84.4724%
Epoch 4/15 - Train Acc: 89.3614%, Train Hard Loss: 0.3560, Train Soft Loss: 0.1171, Train Distill Loss: 0.9931 | Val Hard Loss: 0.7246, Val Soft Loss: 0.2185, Val Distill Loss: 2.1103, Val Acc: 82.2542%
Epoch 5/15 - Train Acc: 92.1863%, Train Hard Loss: 0.2407, Train Soft Loss: 0.0988, Train Distill Loss: 0.7902 | Val Hard Loss: 0.5986, Val Soft Loss: 0.1855, Val Distill Loss: 1.7835, Val

0.9337811900191939

In [13]:
# Initialize the FitNetWrapper with the teacher and student models, and specify the hint and guided layers along with their channel dimensions.
fitnet_wrapper = FitNetWrapper(teacher=densenet_201_teacher, 
                               student=ShuffleNetV2(n_class=10, model_size="0.4x"), 
                               hint_layer_name="features.transition2",
                               guided_layer_name="stage3",
                                student_channels=80,
                                teacher_channels=256,)

# Perform the hint training loop to train the student model using the hints from the teacher model.
trained_fitnet_wrapper = kd_utils.hint_training_loop(fitnet_wrapper, student_train_loader, num_epochs=hint_epochs, device=device)

# After the hint training, we can proceed with the knowledge distillation training loop to further train the student model using the teacher's outputs as soft targets.
optimizer = torch.optim.Adam(trained_fitnet_wrapper.student.parameters(), lr=0.001)
lr_scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=train_epochs, eta_min=0.000001)
trained_shufflenet_v2_student, history = kd_utils.student_training_loop(
   teacher=densenet_201_teacher,
   student=trained_fitnet_wrapper.student,
    train_loader=student_train_loader,
    val_loader=student_val_loader,
    optimizer=optimizer,
    temperature=6,
    alpha=0.80,
    num_epochs=train_epochs,
    device=device,
    scheduler=lr_scheduler,
    save_path=None,
)

kd_utils.evaluate(trained_shufflenet_v2_student, student_test_loader, device)

Overall Progress:   0%|          | 0/3120 [00:00<?, ?it/s]

Epoch 1/15 - Hint Loss: 2.2205
Epoch 2/15 - Hint Loss: 1.7791
Epoch 3/15 - Hint Loss: 1.6210
Epoch 4/15 - Hint Loss: 1.5380
Epoch 5/15 - Hint Loss: 1.4873
Epoch 6/15 - Hint Loss: 1.4530
Epoch 7/15 - Hint Loss: 1.4271
Epoch 8/15 - Hint Loss: 1.4079
Epoch 9/15 - Hint Loss: 1.3937
Epoch 10/15 - Hint Loss: 1.3826
Epoch 11/15 - Hint Loss: 1.3760
Epoch 12/15 - Hint Loss: 1.3684
Epoch 13/15 - Hint Loss: 1.3614
Epoch 14/15 - Hint Loss: 1.3567
Epoch 15/15 - Hint Loss: 1.3526


Overall Progress:   0%|          | 0/3915 [00:00<?, ?it/s]

Epoch 1/15 - Train Acc: 56.3486%, Train Hard Loss: 1.3945, Train Soft Loss: 0.3310, Train Distill Loss: 3.4989 | Val Hard Loss: 1.4112, Val Soft Loss: 0.4230, Val Distill Loss: 4.0900, Val Acc: 64.0288%
Epoch 2/15 - Train Acc: 74.8310%, Train Hard Loss: 0.8645, Train Soft Loss: 0.1881, Train Distill Loss: 2.0459 | Val Hard Loss: 2.0103, Val Soft Loss: 0.5237, Val Distill Loss: 5.1947, Val Acc: 58.9928%
Epoch 3/15 - Train Acc: 83.9820%, Train Hard Loss: 0.5540, Train Soft Loss: 0.1300, Train Distill Loss: 1.3795 | Val Hard Loss: 0.7577, Val Soft Loss: 0.2196, Val Distill Loss: 2.1357, Val Acc: 79.7962%
Epoch 4/15 - Train Acc: 88.7152%, Train Hard Loss: 0.3693, Train Soft Loss: 0.1016, Train Distill Loss: 1.0267 | Val Hard Loss: 0.8142, Val Soft Loss: 0.2006, Val Distill Loss: 2.0118, Val Acc: 82.6739%
Epoch 5/15 - Train Acc: 92.6071%, Train Hard Loss: 0.2279, Train Soft Loss: 0.0800, Train Distill Loss: 0.7586 | Val Hard Loss: 0.5939, Val Soft Loss: 0.1525, Val Distill Loss: 1.5173, Val

0.9395393474088292